# External mapper release sensitivity (pybiomart) — why “time” matters

This notebook is a marketing appendix that makes one core idea *visually undeniable*:

> Identifier mapping is time-dependent. If a tool does not expose a stable “which release?” knob, it is implicitly choosing one for you.

## Rationale

Among common point-in-time mappers, **pybiomart** is one of the few backends that can be asked to query an **Ensembl archive** corresponding to a historical release.
That makes it a good demonstration target for the time axis:

- The *same* input IDs can yield different outputs across releases.
- Even when a mapping is “1→1” at one release, it can become “1→0” or “1→n” at another.

This is **not** an accuracy benchmark. It is a stability / reproducibility demonstration.

## Outputs

- `idtrack-manuscript/figures/fig_external_mapper_pybiomart_release_sensitivity.pdf`
- `idtrack-manuscript/figures/fig_pybiomart_release_sensitivity_instability_by_id.pdf`
- `idtrack-manuscript/tables/pybiomart_release_sensitivity_instability_by_id.csv`
- Cached query results under `idtrack/docs/_notebooks/idtrack_cache/experiments/comparison_release_sensitivity/`

## Notes

- The notebook uses `idtrack._external_mappers.convert_ids` (no ortholog utilities).
- The demo is intentionally small to stay friendly to public services; expand the query list if needed (caching will protect re-runs).


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:  # noqa: S110
    sns = None

import sys

# Add experiments/src to sys.path
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not ((REPO_ROOT / 'idtrack').is_dir() and (REPO_ROOT / 'idtrack-manuscript').is_dir()):
    REPO_ROOT = REPO_ROOT.parent

EXPERIMENTS_SRC = REPO_ROOT / 'idtrack' / 'reproducibility' / 'experiments' / 'src'
sys.path.append(str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    MANUSCRIPT_COLORS,
    notebook_context,
    read_pickle,
    save_figure,
    safe_tag,
    write_pickle,
)

ctx = notebook_context('comparison_release_sensitivity', start=REPO_ROOT)
plt.rcParams.update({'savefig.dpi': 300})

CACHE_DIR = ctx.experiment_cache
MANUSCRIPT_FIGURES = ctx.manuscript_figures

print('Repo root:', REPO_ROOT)
print('CACHE_DIR:', CACHE_DIR)


In [ ]:
# -------------------- Configuration --------------------

METHOD = 'pybiomart'
SPECIES = 'human'

# Focus on manuscript-relevant targets.
INPUT_DB = 'ensembl_gene'
OUTPUT_DB = 'HGNC Symbol'  # or 'UniProtKB/Swiss-Prot'

# Small, interpretable demo set (includes versioned + negative control)
QUERY_IDS = [
    'ENSG00000139618',  # BRCA2
    'ENSG00000141510',  # TP53
    'ENSG00000157764',  # BRAF
    'ENSG00000121879',  # KRAS
    'ENSG00000171862',  # PTEN
    'ENSG00000136997',  # MYC
    'ENSG00000146648',  # EGFR
    'ENSG00000141510.18',  # TP53 (versioned)
    'ENSG_DOES_NOT_EXIST',
]

# Release sweep (choose values that are supported by Ensembl archives).
# Tip: widen this range for a stronger time-axis story; caching will protect re-runs.
RELEASES = [90, 95, 100, 105, 107]

# Backend request knobs
CHUNK_SIZE = 200
PAUSE_S = 0.1
VERBOSE = 2

RESULTS_PKL = CACHE_DIR / (
    f"pybiomart_release_sensitivity_in{safe_tag(INPUT_DB)}_out{safe_tag(OUTPUT_DB)}_"
    f"n{len(QUERY_IDS)}_releases{RELEASES[0]}-{RELEASES[-1]}.pickle"
)

print('METHOD:', METHOD)
print('INPUT_DB:', INPUT_DB)
print('OUTPUT_DB:', OUTPUT_DB)
print('RELEASES:', RELEASES)
print('RESULTS_PKL:', RESULTS_PKL)


In [ ]:
# -------------------- Compute (cache-first) --------------------

import idtrack._external_mappers as ext

status = ext.check_optional_dependencies(warn=True)
if not status.get('pybiomart', False):
    raise RuntimeError("pybiomart backend is not available. Install with: pip install pybiomart")


def outputs_by_input(df: pd.DataFrame, inputs: list[str]) -> dict[str, set[str]]:
    if df is None or df.empty:
        return {str(i): set() for i in inputs}

    out: dict[str, set[str]] = {}
    for inp, sub in df.groupby('input_id'):
        vals = [
            v
            for v in sub['output_id'].tolist()
            if v is not None and str(v).strip() not in {'', 'nan', 'None', 'null'}
        ]
        out[str(inp)] = set(map(str, vals))
    for i in inputs:
        out.setdefault(str(i), set())
    return out


if RESULTS_PKL.exists():
    payload = read_pickle(RESULTS_PKL)
    print('Loaded:', RESULTS_PKL)
else:
    outputs = {}
    for rel in RELEASES:
        df = ext.convert_ids(
            ids=QUERY_IDS,
            input_db=INPUT_DB,
            output_db=OUTPUT_DB,
            method=METHOD,
            species=SPECIES,
            release_for_pybiomart=int(rel),
            chunk_size=int(CHUNK_SIZE),
            pause=float(PAUSE_S),
            verbose=VERBOSE,
        )
        outputs[int(rel)] = outputs_by_input(df, QUERY_IDS)

    payload = {
        'params': {
            'method': METHOD,
            'species': SPECIES,
            'input_db': INPUT_DB,
            'output_db': OUTPUT_DB,
            'releases': RELEASES,
            'n_inputs': len(QUERY_IDS),
        },
        'outputs_by_release': outputs,
    }

    write_pickle(payload, RESULTS_PKL)
    print('Saved:', RESULTS_PKL)

payload['params']


In [ ]:
# -------------------- Analysis + multi-panel figure export --------------------

outputs = payload['outputs_by_release']

# 1) Per-release outcome cardinalities (n_outputs per input)
n_out = pd.DataFrame(index=[str(q) for q in QUERY_IDS], columns=[int(r) for r in RELEASES], dtype=int)
for rel in RELEASES:
    for q in QUERY_IDS:
        n_out.loc[str(q), int(rel)] = len(outputs[int(rel)].get(str(q), set()))

# 2) Fraction of inputs that change vs a reference release
ref = int(RELEASES[-1])
changed_frac = {}
for rel in RELEASES:
    rel = int(rel)
    changed = [outputs[rel].get(str(q), set()) != outputs[ref].get(str(q), set()) for q in QUERY_IDS]
    changed_frac[rel] = float(np.mean(changed)) if changed else float('nan')

# 3) Release x release agreement (mean Jaccard across queries)
def jaccard(a: set[str], b: set[str]) -> float:
    denom = len(a | b)
    return (len(a & b) / denom) if denom else 1.0

jmat = pd.DataFrame(index=[int(r) for r in RELEASES], columns=[int(r) for r in RELEASES], dtype=float)
for r1 in RELEASES:
    for r2 in RELEASES:
        vals = [jaccard(outputs[int(r1)].get(str(q), set()), outputs[int(r2)].get(str(q), set())) for q in QUERY_IDS]
        jmat.loc[int(r1), int(r2)] = float(np.mean(vals)) if vals else float('nan')

fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.5), constrained_layout=True)
ax0, ax1, ax2 = axes

# Panel A: changed fraction
ax0.plot(list(changed_frac.keys()), list(changed_frac.values()), '-o', color=MANUSCRIPT_COLORS['1→1'])
ax0.set_ylim(0, 1)
ax0.set_xlabel('pybiomart release')
ax0.set_ylabel('Fraction of inputs with changed output set')
ax0.set_title('Release sensitivity (set changes vs reference)')

# Panel B: release-by-release agreement
if sns is not None:
    sns.heatmap(jmat, ax=ax1, cmap='Blues', vmin=0, vmax=1, square=True, cbar_kws={'label': 'Mean Jaccard'})
else:
    im = ax1.imshow(jmat.values, cmap='Blues', vmin=0, vmax=1)
    fig.colorbar(im, ax=ax1, label='Mean Jaccard')
    ax1.set_xticks(range(len(jmat.columns)))
    ax1.set_yticks(range(len(jmat.index)))
    ax1.set_xticklabels(jmat.columns, rotation=30, ha='right')
    ax1.set_yticklabels(jmat.index)
ax1.set_title('Agreement across releases')
ax1.set_xlabel('Release')
ax1.set_ylabel('Release')

# Panel C: per-input output cardinality
if sns is not None:
    sns.heatmap(n_out, ax=ax2, cmap='Greys', cbar_kws={'label': '# outputs'})
else:
    im2 = ax2.imshow(n_out.values, cmap='Greys')
    fig.colorbar(im2, ax=ax2, label='# outputs')
    ax2.set_xticks(range(len(n_out.columns)))
    ax2.set_yticks(range(len(n_out.index)))
    ax2.set_xticklabels(n_out.columns, rotation=30, ha='right')
    ax2.set_yticklabels(n_out.index)
ax2.set_title('Output cardinality (n per input)')
ax2.set_xlabel('Release')
ax2.set_ylabel('Input ID')

written = save_figure(fig, 'fig_external_mapper_pybiomart_release_sensitivity.pdf', ctx, formats=('pdf',))
print('Saved:', written['pdf'])

n_out


In [ ]:
# -------------------- Export summary table (CSV) --------------------

rows = []
ref = int(RELEASES[-1])
for rel in RELEASES:
    rel = int(rel)
    for q in QUERY_IDS:
        a = outputs[rel].get(str(q), set())
        b = outputs[ref].get(str(q), set())
        rows.append(
            {
                'release': rel,
                'input_id': str(q),
                'n_outputs': len(a),
                'outputs': ';'.join(sorted(a)[:30]),
                'changed_vs_ref': a != b,
                'ref_release': ref,
            }
        )

df = pd.DataFrame(rows)
out_csv = CACHE_DIR / 'pybiomart_release_sensitivity_outputs.csv'
df.to_csv(out_csv, index=False)
print('Wrote:', out_csv)

df.head(10)


# Marketing extension: per-identifier instability score

A strong time-axis story is not just “the aggregate changes”, but:

> the *same input identifier* can map to different outputs depending on which “current” release the mapper implicitly uses.

This section computes a per-input instability score:

- how many **distinct output sets** are observed across the release sweep
- whether the input is **ever unmapped** at some release


In [ ]:
from experiments_utils import atomic_write_dataframe_csv, label_panels  # noqa: E402

rows = []
for q in QUERY_IDS:
    sets = [frozenset(outputs[int(r)].get(str(q), set())) for r in RELEASES]
    rows.append(
        {
            'input_id': str(q),
            'n_distinct_output_sets': len(set(sets)),
            'ever_unmapped': any(len(s) == 0 for s in sets),
            'max_n_outputs': max((len(s) for s in sets), default=0),
        }
    )

inst = pd.DataFrame(rows).sort_values(['n_distinct_output_sets', 'ever_unmapped'], ascending=[False, False])
out_inst = ctx.manuscript_tables / 'pybiomart_release_sensitivity_instability_by_id.csv'
atomic_write_dataframe_csv(inst, out_inst, index=False)
atomic_write_dataframe_csv(inst, ctx.experiment_outputs / 'tables' / out_inst.name, index=False)
print('Wrote:', out_inst)
display(inst)

figI, axesI = plt.subplots(1, 2, figsize=(12.5, 4.2), constrained_layout=True)
ax0, ax1 = axesI

ax0.barh(inst['input_id'], inst['n_distinct_output_sets'], color=MANUSCRIPT_COLORS['1→n'])
ax0.set_xlabel('# distinct output sets across releases')
ax0.set_ylabel('input_id')
ax0.set_title('Per-input mapping instability')

ax1.barh(inst['input_id'], inst['max_n_outputs'], color=MANUSCRIPT_COLORS['1→1'])
ax1.set_xlabel('max # outputs at any release')
ax1.set_ylabel('')
ax1.set_title('Max ambiguity observed')

label_panels(axesI)
writtenI = save_figure(figI, 'fig_pybiomart_release_sensitivity_instability_by_id.pdf', ctx, formats=('pdf',))
print('Saved:', writtenI['pdf'])
